In [ ]:
#!pip install torch scikit-learn pandas numpy matplotlib seaborn --quiet
#!pip install 'rtdl==0.0.13' --no-deps --quiet   # --no-deps avoids conflict
print('Done')

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, math, time
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import rtdl

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, matthews_corrcoef, cohen_kappa_score,
    average_precision_score, precision_recall_curve
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'rtdl version: {rtdl.__version__}')

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})
PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']
print('Imports OK')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
DATA_PATH    = '..data/SAML-D.csv'
N_SPLITS     = 5
RANDOM_STATE = 42
BATCH_SIZE   = 1024
N_EPOCHS     = 30          # increase to 50-100 for better results
LR           = 1e-4
WEIGHT_DECAY = 1e-5
PATIENCE     = 7           # early stopping patience

# FT-Transformer backbone (make_default with n_blocks=3)
N_BLOCKS     = 3

# XGBoost results to compare against (paste your OOF results here)
XGB_RESULTS = {
    'Model'    : 'XGBoost',
    'Accuracy' : 0.7900,   # ← replace with your actual values
    'F1-Score' : 0.7600,
    'AUC-ROC'  : 0.9200,
    'MCC'      : 0.7500,
}
print('Config set')

In [ ]:
# ── LOAD & PREPROCESS ─────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, on_bad_lines='skip', engine='python', encoding='utf-8')
df.columns = df.columns.str.strip()

# Auto-detect target
possible = [c for c in df.columns if c.lower() in ['label','labels','class','target','attack','category']]
TARGET_COL = possible[0] if possible else df.columns[-1]
print(f'Target column: "{TARGET_COL}"  |  Shape: {df.shape}')

# Drop NaN targets
df = df.dropna(subset=[TARGET_COL])

# Identify numeric vs categorical BEFORE encoding target
cat_cols = [c for c in df.select_dtypes(include=['object','category']).columns if c != TARGET_COL]
num_cols  = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COL]

# Encode categorical features
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Encode target
le_target = LabelEncoder()
df[TARGET_COL] = le_target.fit_transform(df[TARGET_COL].astype(str))

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Fill NaN features with median
X = X.fillna(X.median(numeric_only=True))

# Re-encode y to guarantee 0..N-1 contiguous (same fix as XGBoost)
le_final = LabelEncoder()
y = pd.Series(le_final.fit_transform(y), index=y.index)

ALL_CLASSES  = sorted(y.unique())
N_CLASSES    = len(ALL_CLASSES)
IS_BINARY    = (N_CLASSES == 2)
AVERAGE      = 'binary' if IS_BINARY else 'weighted'
CLASS_NAMES  = [str(c) for c in le_target.classes_]

# All features are treated as numerical for FT-Transformer
# (categoricals already label-encoded to integers)
ALL_NUM_COLS = X.columns.tolist()
N_NUM        = len(ALL_NUM_COLS)

print(f'Features: {N_NUM}  |  Classes: {N_CLASSES}  |  Samples: {len(X):,}')
print(f'Class distribution:\n{y.value_counts().sort_index()}')

In [ ]:
# ── FT-TRANSFORMER BUILDER ────────────────────────────────────────────────
def build_ft_transformer(n_num_features, n_classes, n_blocks=3):
    """Build FTTransformer using make_default (paper-recommended config)."""
    d_out = 1 if n_classes == 2 else n_classes
    model = rtdl.FTTransformer.make_default(
        n_num_features   = n_num_features,
        cat_cardinalities= None,        # all features already numeric
        d_out            = d_out,
        n_blocks         = n_blocks,
    )
    return model.to(DEVICE)


def get_loss_fn(n_classes):
    if n_classes == 2:
        return nn.BCEWithLogitsLoss()
    return nn.CrossEntropyLoss()


def predict_proba(model, X_tensor, n_classes, batch_size=2048):
    """Run inference in batches, return numpy proba array."""
    model.eval()
    all_logits = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch_size):
            xb = X_tensor[i:i+batch_size].to(DEVICE)
            logits = model(xb, None)      # no categorical input
            all_logits.append(logits.cpu())
    logits = torch.cat(all_logits, dim=0)
    if n_classes == 2:
        proba_pos = torch.sigmoid(logits.squeeze(-1)).numpy()
        return np.stack([1 - proba_pos, proba_pos], axis=1)
    else:
        return torch.softmax(logits, dim=1).numpy()


print('FT-Transformer builder ready')
# Quick sanity check
_tmp = build_ft_transformer(N_NUM, N_CLASSES, N_BLOCKS)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')
del _tmp

In [ ]:
# ── STRATIFIED K-FOLD TRAINING ────────────────────────────────────────────
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_results  = []
fold_cms      = []
fold_fpr, fold_tpr, fold_auc = [], [], []
fold_pr_prec, fold_pr_rec, fold_ap = [], [], []
fold_train_loss = []    # loss curves per fold
fold_val_loss   = []

oof_proba = np.zeros((len(y), N_CLASSES))
oof_pred  = np.zeros(len(y), dtype=int)

X_np = X.values.astype(np.float32)
y_np = y.values

print(f'Starting {N_SPLITS}-Fold CV  |  Device: {DEVICE}  |  Classes: {N_CLASSES}')
print('='*70)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_np, y_np), 1):
    t_start = time.time()
    print(f'\nFOLD {fold}/{N_SPLITS}')
    print(f'  Train: {len(train_idx):,}  |  Test: {len(test_idx):,}')

    X_train_np, X_test_np = X_np[train_idx], X_np[test_idx]
    y_train_np, y_test_np = y_np[train_idx], y_np[test_idx]

    # ── Scale features (fit on train only) ───────────────────────────────
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train_np).astype(np.float32)
    X_test_sc  = scaler.transform(X_test_np).astype(np.float32)

    # ── Tensors ───────────────────────────────────────────────────────────
    X_tr = torch.tensor(X_train_sc)
    X_te = torch.tensor(X_test_sc)

    if IS_BINARY:
        y_tr = torch.tensor(y_train_np, dtype=torch.float32)
        y_te = torch.tensor(y_test_np,  dtype=torch.float32)
    else:
        y_tr = torch.tensor(y_train_np, dtype=torch.long)
        y_te = torch.tensor(y_test_np,  dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=False
    )
    val_loader = DataLoader(
        TensorDataset(X_te, y_te),
        batch_size=BATCH_SIZE*2, shuffle=False
    )

    # ── Model, optimizer, loss ───────────────────────────────────────────
    model    = build_ft_transformer(N_NUM, N_CLASSES, N_BLOCKS)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    loss_fn   = get_loss_fn(N_CLASSES)

    best_val_loss = float('inf')
    patience_ctr  = 0
    best_state    = None
    epoch_train_losses = []
    epoch_val_losses   = []

    # ── Training loop ─────────────────────────────────────────────────────
    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        batch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb, None)
            if IS_BINARY:
                loss = loss_fn(logits.squeeze(-1), yb)
            else:
                loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        scheduler.step()

        # Validation loss
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb, None)
                if IS_BINARY:
                    vloss = loss_fn(logits.squeeze(-1), yb)
                else:
                    vloss = loss_fn(logits, yb)
                val_losses.append(vloss.item())

        tr_loss  = np.mean(batch_losses)
        val_loss = np.mean(val_losses)
        epoch_train_losses.append(tr_loss)
        epoch_val_losses.append(val_loss)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr  = 0
        else:
            patience_ctr += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{N_EPOCHS}  '
                  f'train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  '
                  f'patience={patience_ctr}/{PATIENCE}')

        if patience_ctr >= PATIENCE:
            print(f'  Early stop at epoch {epoch}')
            break

    fold_train_loss.append(epoch_train_losses)
    fold_val_loss.append(epoch_val_losses)

    # Load best weights
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    # ── Inference ─────────────────────────────────────────────────────────
    proba = predict_proba(model, X_te, N_CLASSES)
    preds = np.argmax(proba, axis=1)

    oof_proba[test_idx] = proba
    oof_pred[test_idx]  = preds

    y_test_np_fold = y_test_np

    # ── Metrics ───────────────────────────────────────────────────────────
    acc  = accuracy_score(y_test_np_fold, preds)
    prec = precision_score(y_test_np_fold, preds, average=AVERAGE, zero_division=0)
    rec  = recall_score(y_test_np_fold, preds, average=AVERAGE, zero_division=0)
    f1   = f1_score(y_test_np_fold, preds, average=AVERAGE, zero_division=0)
    mcc  = matthews_corrcoef(y_test_np_fold, preds)
    kap  = cohen_kappa_score(y_test_np_fold, preds)

    classes_in_fold = sorted(np.unique(y_test_np_fold))
    proba_sub = proba[:, classes_in_fold]
    row_sums  = proba_sub.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    proba_sub = proba_sub / row_sums

    try:
        if IS_BINARY:
            auc = roc_auc_score(y_test_np_fold, proba[:, 1])
            ap  = average_precision_score(y_test_np_fold, proba[:, 1])
            fpr, tpr, _ = roc_curve(y_test_np_fold, proba[:, 1])
            pr_p, pr_r, _ = precision_recall_curve(y_test_np_fold, proba[:, 1])
            fold_fpr.append(fpr); fold_tpr.append(tpr)
            fold_pr_prec.append(pr_p); fold_pr_rec.append(pr_r)
        else:
            auc = roc_auc_score(
                y_test_np_fold, proba_sub,
                multi_class='ovr', average='weighted',
                labels=classes_in_fold
            )
            ap = average_precision_score(
                pd.get_dummies(pd.Series(y_test_np_fold)).values,
                proba_sub, average='weighted'
            )
    except ValueError as e:
        print(f'  [AUC skipped] {e}')
        auc, ap = float('nan'), float('nan')

    cm = confusion_matrix(y_test_np_fold, preds, labels=ALL_CLASSES)
    elapsed = time.time() - t_start

    fold_results.append({
        'Fold': fold, 'Train Size': len(train_idx), 'Test Size': len(test_idx),
        'Accuracy': round(acc,4), 'Precision': round(prec,4),
        'Recall': round(rec,4), 'F1-Score': round(f1,4),
        'AUC-ROC': round(auc,4), 'Avg Precision': round(ap,4),
        'MCC': round(mcc,4), 'Kappa': round(kap,4),
        'Time(s)': round(elapsed,1),
    })
    fold_cms.append(cm)
    fold_auc.append(auc)
    fold_ap.append(ap)

    print(f'  ── Results ──────────────────────────')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}   Recall : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}   AUC-ROC: {auc:.4f}')
    print(f'  Avg Prec  : {ap:.4f}   MCC    : {mcc:.4f}   Kappa: {kap:.4f}')
    print(f'  Time      : {elapsed:.1f}s')
    print(f'\n  Classification Report (Fold {fold}):')
    print(classification_report(
        y_test_np_fold, preds,
        labels=ALL_CLASSES, target_names=CLASS_NAMES, zero_division=0
    ))
    print('='*70)
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ── SUMMARY TABLE ─────────────────────────────────────────────────────────
results_df  = pd.DataFrame(fold_results)
metric_cols = ['Accuracy','Precision','Recall','F1-Score','AUC-ROC','Avg Precision','MCC','Kappa']

mean_row = {'Fold': 'MEAN', **{c: round(results_df[c].mean(),4) for c in metric_cols}}
std_row  = {'Fold': 'STD',  **{c: round(results_df[c].std(), 4) for c in metric_cols}}
summary_df = pd.concat([results_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print('\n' + '='*70)
print('CROSS-VALIDATION SUMMARY — FT-Transformer')
print('='*70)
print(summary_df[['Fold']+metric_cols].to_string(index=False))
print('='*70)

In [ ]:
# ── VIZ 1: Per-Fold Metrics Bar Chart ────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Per-Fold Metric Breakdown — FT-Transformer on SAML-D',
             fontsize=14, fontweight='bold', y=1.01)
folds = results_df['Fold'].tolist()

for ax, metric in zip(axes.flat, metric_cols):
    vals = results_df[metric].tolist()
    bars = ax.bar(folds, vals, color=PALETTE, edgecolor='white', zorder=3)
    mean_val = np.mean(vals)
    ax.axhline(mean_val, color='black', linestyle='--', lw=1.3,
               label=f'Mean={mean_val:.4f}')
    ax.set_title(metric, fontweight='bold')
    ax.set_xlabel('Fold'); ax.set_ylabel('Score')
    ax.set_xticks(folds); ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3, zorder=0)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_ylim(max(0, min(vals)-0.05), min(1, max(vals)+0.06))

plt.tight_layout()
plt.savefig('ftt_fold_metrics_bar.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── VIZ 2: Metrics Heatmap ────────────────────────────────────────────────
heat_data = results_df.set_index('Fold')[metric_cols].astype(float)
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heat_data.T, annot=True, fmt='.4f', cmap='YlGnBu',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Score'})
ax.set_title('Metrics Heatmap — Each Fold (FT-Transformer)', fontweight='bold', pad=12)
ax.set_xlabel('Fold'); ax.set_ylabel('Metric')
plt.tight_layout()
plt.savefig('ftt_metrics_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── VIZ 3: Training & Validation Loss Curves ─────────────────────────────
fig, axes = plt.subplots(1, N_SPLITS, figsize=(5*N_SPLITS, 4), sharey=False)
fig.suptitle('Train vs Val Loss per Fold — FT-Transformer', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes):
    tr = fold_train_loss[i]
    vl = fold_val_loss[i]
    epochs = range(1, len(tr)+1)
    ax.plot(epochs, tr, label='Train', color=PALETTE[0], lw=2)
    ax.plot(epochs, vl, label='Val',   color=PALETTE[1], lw=2)
    best_ep = np.argmin(vl) + 1
    ax.axvline(best_ep, color='grey', linestyle=':', lw=1.5,
               label=f'Best ep={best_ep}')
    ax.set_title(f'Fold {i+1}', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('ftt_loss_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── VIZ 4: Confusion Matrices ─────────────────────────────────────────────
fig, axes = plt.subplots(1, N_SPLITS, figsize=(5*N_SPLITS, 5))
fig.suptitle('Confusion Matrices per Fold — FT-Transformer', fontsize=14, fontweight='bold')

for i, (ax, cm) in enumerate(zip(axes, fold_cms), 1):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.3, cbar=False)
    ax.set_title(f'Fold {i}  (AUC={fold_auc[i-1]:.3f})', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('ftt_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── VIZ 5: ROC + PR Curves (binary only) ──────────────────────────────────
if IS_BINARY:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # ROC
    mean_fpr = np.linspace(0, 1, 200)
    interp_tprs = []
    for i in range(N_SPLITS):
        ax1.plot(fold_fpr[i], fold_tpr[i],
                 color=PALETTE[i%len(PALETTE)], lw=1.5, alpha=0.8,
                 label=f'Fold {i+1}  AUC={fold_auc[i]:.4f}')
        interp_tprs.append(np.interp(mean_fpr, fold_fpr[i], fold_tpr[i]))
    mean_tpr = np.mean(interp_tprs, axis=0)
    std_tpr  = np.std(interp_tprs, axis=0)
    mean_auc = np.mean(fold_auc)
    ax1.plot(mean_fpr, mean_tpr, 'k--', lw=2.5,
             label=f'Mean AUC={mean_auc:.4f} ± {np.std(fold_auc):.4f}')
    ax1.fill_between(mean_fpr, mean_tpr-std_tpr, mean_tpr+std_tpr, alpha=0.15, color='grey')
    ax1.plot([0,1],[0,1],'k:',lw=1)
    ax1.set(xlabel='FPR', ylabel='TPR', title='ROC Curves — All Folds')
    ax1.legend(fontsize=9, loc='lower right'); ax1.grid(alpha=0.3)

    # PR
    for i in range(N_SPLITS):
        ax2.plot(fold_pr_rec[i], fold_pr_prec[i],
                 color=PALETTE[i%len(PALETTE)], lw=1.5, alpha=0.8,
                 label=f'Fold {i+1}  AP={fold_ap[i]:.4f}')
    ax2.set(xlabel='Recall', ylabel='Precision',
            title=f'PR Curves (Mean AP={np.mean(fold_ap):.4f})')
    ax2.legend(fontsize=9, loc='upper right'); ax2.grid(alpha=0.3)

    plt.suptitle('ROC & Precision-Recall — FT-Transformer', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('ftt_roc_pr.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print(f'Multiclass — Mean weighted AUC-ROC: {np.mean(fold_auc):.4f} ± {np.std(fold_auc):.4f}')

In [ ]:
# ── VIZ 6: Metric Boxplot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
metric_matrix = results_df[metric_cols].values.T

bp = ax.boxplot(metric_matrix.T, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], sns.color_palette('Set2', len(metric_cols))):
    patch.set_facecolor(color); patch.set_alpha(0.7)

for i, vals in enumerate(metric_matrix, 1):
    jitter = np.random.uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(np.full(len(vals), i)+jitter, vals, color='black', s=30, zorder=5)

ax.set_xticks(range(1, len(metric_cols)+1))
ax.set_xticklabels(metric_cols, rotation=30, ha='right')
ax.set_ylabel('Score')
ax.set_title('Metric Distribution Across Folds — FT-Transformer', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('ftt_metric_boxplot.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── VIZ 7: Radar Charts ───────────────────────────────────────────────────
radar_metrics = ['Accuracy','Precision','Recall','F1-Score','AUC-ROC','MCC']

def radar_chart(ax, values, labels, color, title):
    N = len(labels)
    angles = [n / N * 2 * np.pi for n in range(N)] + [0]
    values = list(values) + [values[0]]
    ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, size=8)
    ax.set_ylim(0, 1)
    ax.plot(angles, values, color=color, lw=2)
    ax.fill(angles, values, color=color, alpha=0.2)
    ax.set_title(title, size=10, fontweight='bold', pad=12)

fig = plt.figure(figsize=(18, 9))
fig.suptitle('Radar Charts — Per-Fold Profile (FT-Transformer)', fontsize=14, fontweight='bold')

for i in range(N_SPLITS):
    ax = fig.add_subplot(2, 3, i+1, polar=True)
    vals = [results_df.loc[i, m] for m in radar_metrics]
    vals[-1] = (vals[-1] + 1) / 2   # normalize MCC to [0,1]
    radar_chart(ax, vals, radar_metrics, PALETTE[i%len(PALETTE)], f'Fold {i+1}')

ax = fig.add_subplot(2, 3, N_SPLITS+1, polar=True)
mean_vals = [results_df[m].mean() for m in radar_metrics]
mean_vals[-1] = (mean_vals[-1] + 1) / 2
radar_chart(ax, mean_vals, radar_metrics, 'black', 'MEAN')
plt.tight_layout()
plt.savefig('ftt_radar.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── OOF EVALUATION ───────────────────────────────────────────────────────
print('\n' + '='*70)
print('OUT-OF-FOLD (OOF) OVERALL EVALUATION — FT-Transformer')
print('='*70)

predicted_mask = oof_proba.sum(axis=1) > 0
y_eval         = y_np[predicted_mask]
pred_eval      = oof_pred[predicted_mask]
proba_eval     = oof_proba[predicted_mask]

# Normalize probas
row_sums = proba_eval.sum(axis=1, keepdims=True)
row_sums[row_sums==0] = 1
proba_norm = proba_eval / row_sums

print(classification_report(y_eval, pred_eval,
                             labels=ALL_CLASSES, target_names=CLASS_NAMES, zero_division=0))

oof_acc = accuracy_score(y_eval, pred_eval)
oof_f1  = f1_score(y_eval, pred_eval, average=AVERAGE, zero_division=0)
oof_mcc = matthews_corrcoef(y_eval, pred_eval)

classes_present = sorted(np.unique(y_eval))
try:
    if IS_BINARY:
        oof_auc = roc_auc_score(y_eval, proba_norm[:,1])
    else:
        oof_auc = roc_auc_score(
            y_eval, proba_norm[:, classes_present],
            multi_class='ovr', average='weighted', labels=classes_present
        )
except ValueError as e:
    print(f'OOF AUC error: {e}'); oof_auc = float('nan')

print(f'OOF Accuracy : {oof_acc:.4f}')
print(f'OOF F1-Score : {oof_f1:.4f}')
print(f'OOF AUC-ROC  : {oof_auc:.4f}')
print(f'OOF MCC      : {oof_mcc:.4f}')

fig, ax = plt.subplots(figsize=(14, 11))
oof_cm = confusion_matrix(y_eval, pred_eval, labels=ALL_CLASSES)
sns.heatmap(oof_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, linewidths=0.3)
ax.set_title(f'OOF Confusion Matrix — FT-Transformer  (AUC={oof_auc:.4f})', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('ftt_oof_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── VIZ 8: XGBoost vs FT-Transformer Comparison ───────────────────────────
compare_metrics = ['Accuracy','F1-Score','AUC-ROC','MCC']

ftt_results = {
    'Model'    : 'FT-Transformer',
    'Accuracy' : round(oof_acc, 4),
    'F1-Score' : round(oof_f1,  4),
    'AUC-ROC'  : round(oof_auc, 4),
    'MCC'      : round(oof_mcc, 4),
}

comp_df = pd.DataFrame([XGB_RESULTS, ftt_results])
print('\n' + '='*70)
print('MODEL COMPARISON — XGBoost vs FT-Transformer')
print('='*70)
print(comp_df.to_string(index=False))
print('='*70)

# Bar chart
x   = np.arange(len(compare_metrics))
w   = 0.35
fig, ax = plt.subplots(figsize=(10, 6))

xgb_vals = [XGB_RESULTS[m] for m in compare_metrics]
ftt_vals = [ftt_results[m]  for m in compare_metrics]

b1 = ax.bar(x - w/2, xgb_vals, w, label='XGBoost',        color='#4C72B0', edgecolor='white')
b2 = ax.bar(x + w/2, ftt_vals, w, label='FT-Transformer',  color='#DD8452', edgecolor='white')

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(compare_metrics, fontsize=12)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.12)
ax.set_title('XGBoost vs FT-Transformer — OOF Metric Comparison',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('xgb_vs_ftt_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

# Radar comparison
fig = plt.figure(figsize=(12, 6))
fig.suptitle('Radar: XGBoost vs FT-Transformer', fontsize=14, fontweight='bold')

for idx, (label, vals_dict, color) in enumerate([
    ('XGBoost',       XGB_RESULTS, '#4C72B0'),
    ('FT-Transformer', ftt_results, '#DD8452'),
]):
    ax = fig.add_subplot(1, 2, idx+1, polar=True)
    vals = [vals_dict[m] for m in compare_metrics]
    radar_chart(ax, vals, compare_metrics, color, label)

plt.tight_layout()
plt.savefig('xgb_vs_ftt_radar.png', bbox_inches='tight', dpi=150)
plt.show()